# 01 - Data and DVC Setup

Run this after `00_colab_setup.ipynb`. It mounts Google Drive, configures a Drive-backed DVC cache, pulls existing DVC data if available, and adds `labeled_dataset.csv` to DVC when you have placed it in Drive. It does not download or unzip the full Cap3D archive automatically.

In [ ]:
REPO_DIR = "/content/graphcnn-federated-3d"
DRIVE_ROOT = "/content/drive/MyDrive/graphcnn-federated-3d"
DVC_STORE = f"{DRIVE_ROOT}/dvc-store"
DRIVE_METADATA_CSV = f"{DRIVE_ROOT}/labeled_dataset.csv"

print("Repo dir:", REPO_DIR)
print("Drive project folder:", DRIVE_ROOT)
print("DVC store:", DVC_STORE)
print("Expected Drive metadata CSV:", DRIVE_METADATA_CSV)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import shutil

%cd {REPO_DIR}
Path(DRIVE_ROOT).mkdir(parents=True, exist_ok=True)
Path(DVC_STORE).mkdir(parents=True, exist_ok=True)
Path("data/metadata").mkdir(parents=True, exist_ok=True)
Path("data/raw").mkdir(parents=True, exist_ok=True)
Path("data/processed").mkdir(parents=True, exist_ok=True)
Path("data/splits").mkdir(parents=True, exist_ok=True)
Path("data/cache").mkdir(parents=True, exist_ok=True)

drive_metadata = Path(DRIVE_METADATA_CSV)
repo_metadata = Path("data/metadata/labeled_dataset.csv")
if drive_metadata.exists() and not repo_metadata.exists():
    shutil.copy2(drive_metadata, repo_metadata)
    print(f"Copied {drive_metadata} -> {repo_metadata}")
elif repo_metadata.exists():
    print(f"Metadata already exists at {repo_metadata}")
else:
    print(f"Optional step: upload labeled_dataset.csv to {drive_metadata}")

In [ ]:
!bash scripts/setup_dvc.sh
!dvc remote add -d drive_store {DVC_STORE} --local --force
!dvc remote list

In [ ]:
# Pull any DVC-tracked files that already exist in the Drive store.
# This is allowed to fail on the first run before anything has been pushed.
!dvc pull || echo "No DVC data pulled yet. This is normal before the first dvc push."

In [ ]:
from pathlib import Path

metadata_path = Path("data/metadata/labeled_dataset.csv")
if metadata_path.exists():
    !dvc add data/metadata/labeled_dataset.csv
    !dvc push
    print("Metadata is now in the Drive-backed DVC store.")
    print("Commit these pointer files from your local machine or Colab:")
    print("  data/metadata/labeled_dataset.csv.dvc")
    print("  data/metadata/.gitignore")
else:
    print("No labeled_dataset.csv found. Upload it to Drive, rerun this notebook, then commit the .dvc pointer files.")

Cap3D download is intentionally manual. Start with metadata and a small subset; avoid unzipping the full archive blindly on Colab.

In [ ]:
DOWNLOAD_CAP3D_SAMPLE_FILES = False

if DOWNLOAD_CAP3D_SAMPLE_FILES:
    !python scripts/download_cap3d.py --data-dir data/raw
else:
    print("Skipped Cap3D download. Set DOWNLOAD_CAP3D_SAMPLE_FILES=True only when you are ready.")